# Eigenvalues and Eigenvectors

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.02, 01.05 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/06_eigenvalues_eigenvectors.ipynb)

---

## 🎯 Learning Objective

Learn what eigenvalues and eigenvectors reveal about a linear transformation's natural axes, and understand positive-definiteness and condition number as diagnostics for optimization.

---

## 📐 Theory

An **eigenvector** of a square matrix $A$ is a nonzero vector $\mathbf{v}$ whose direction
$A$ does not change — $A$ only stretches or shrinks it by a scalar factor $\lambda$, its
**eigenvalue**:

$$A\mathbf{v} = \lambda\mathbf{v}$$

Every other vector gets rotated *and* scaled by $A$; eigenvectors are the special directions
that only get scaled. Finding them means solving $\det(A - \lambda I) = 0$ — the
**characteristic equation**.

### Eigendecomposition and the spectral theorem

If $A$ (size $n\times n$) has $n$ linearly independent eigenvectors, it can be written as

$$A = V\Lambda V^{-1}$$

where columns of $V$ are the eigenvectors and $\Lambda$ is diagonal with the eigenvalues. This
**eigendecomposition** re-expresses $A$'s action as "change basis to the eigenvectors, scale
each axis independently, change back."

The **spectral theorem** gives a much nicer guarantee for **symmetric** matrices ($A=A^\top$,
which covariance matrices always are): their eigenvectors can always be chosen orthogonal, so
$A = Q\Lambda Q^\top$ with $Q$ orthogonal ($Q^\top Q = I$) — decomposition without ever needing
a numerically fragile inverse.

### Positive definiteness

A symmetric matrix $A$ is **positive definite** if $\mathbf{x}^\top A\mathbf{x} > 0$ for every
nonzero $\mathbf{x}$ — equivalently, every eigenvalue of $A$ is strictly positive.
Positive-*semi*-definite allows eigenvalues of exactly zero. This matters constantly: the
Hessian of a loss function being positive definite at a point is exactly the condition for
that point to be a local minimum (not a saddle point).

### Condition number

$$\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

(ratio of largest to smallest eigenvalue/singular value magnitude). A large condition number
means $A$ stretches some directions vastly more than others — exactly the "nearly singular"
sensitivity we saw break numerical stability in
[Notebook 01.05](05_determinants_and_inverses.ipynb).

---

## 👁️ Visual Intuition

Eigenvectors are the only directions a transformation doesn't rotate. Overlaying eigenvector
arrows on a transformed grid of points shows them staying put in direction while everything
else swings around.

In [ ]:
# Setup + eigenvectors as the "unrotated" directions of a linear map
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

A = np.array([[3.0, 1.0], [0.0, 2.0]])
eigvals, eigvecs = np.linalg.eig(A)

theta = np.linspace(0, 2 * np.pi, 60)
circle = np.stack([np.cos(theta), np.sin(theta)])
transformed_circle = A @ circle

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*transformed_circle, color="#4C72B0", label="A @ unit circle")
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    ax.quiver(0, 0, v[0], v[1], color="green", angles='xy', scale_units='xy', scale=1,
              width=0.01, label=f"eigenvector {i+1} ($\\lambda$={lam:.1f})")
    ax.quiver(0, 0, lam * v[0], lam * v[1], color="red", angles='xy', scale_units='xy', scale=1,
              width=0.008, alpha=0.6)
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.set_title("Eigenvectors (green) keep their direction; red shows A applied to them")
ax.legend(fontsize=8)
plt.show()

## 🐍 Implementation from Scratch

The power iteration method finds the dominant eigenvector using nothing but repeated
matrix-vector multiplication — a good way to *feel* what "the direction A stretches most"
means, and the same core idea that scales up to huge matrices where full eigendecomposition
would be too expensive.

In [ ]:
# Power iteration: find the dominant eigenvector/eigenvalue via repeated multiplication
import numpy as np

def power_iteration(A, n_iter=100):
    n = A.shape[0]
    v = np.random.default_rng(0).standard_normal(n)
    v = v / np.linalg.norm(v)
    for _ in range(n_iter):
        v = A @ v
        v = v / np.linalg.norm(v)             # renormalize each step to avoid overflow/underflow
    eigenvalue = v @ A @ v                     # Rayleigh quotient (v is now a unit eigenvector)
    return eigenvalue, v

A = np.array([[4.0, 1.0], [2.0, 3.0]])
lam_power, v_power = power_iteration(A)

eigvals_np, eigvecs_np = np.linalg.eig(A)
dominant_idx = np.argmax(np.abs(eigvals_np))

print("Power iteration eigenvalue :", round(lam_power, 4))
print("np.linalg.eig dominant     :", round(eigvals_np[dominant_idx].real, 4))
print("Power iteration eigenvector:", np.round(v_power, 4))
print("np.linalg.eig eigenvector  :", np.round(eigvecs_np[:, dominant_idx].real, 4), "(up to sign)")

# Positive-definiteness check via eigenvalues
def is_positive_definite(A):
    eigvals = np.linalg.eigvalsh(A)   # eigvalsh assumes/exploits symmetry, returns REAL eigenvalues
    return np.all(eigvals > 0)

cov_like = np.array([[2.0, 0.5], [0.5, 1.0]])
print("\nIs a covariance-like matrix positive definite?", is_positive_definite(cov_like))
print("Condition number (lambda_max / lambda_min):", round(np.linalg.cond(cov_like), 4))

## 🤖 Why This Matters for AI

**PCA is an eigendecomposition** of the data covariance matrix: the eigenvectors are the
principal directions of variance, and eigenvalues tell you how much variance each direction
explains. Below we run PCA "by hand" via eigendecomposition and confirm it matches
scikit-learn, previewing [Notebook 01.07](07_SVD_and_PCA.ipynb) which does the same thing via
SVD (the numerically preferred route in practice).

**Condition number and training stability**: the condition number of the loss Hessian governs
how fast (or slowly, or unstably) gradient descent converges — a running theme picked up again
in Module 04 (Optimization).

In [ ]:
# PCA "by hand" via eigendecomposition of the covariance matrix
import numpy as np
from sklearn.decomposition import PCA

rng = np.random.default_rng(3)
# Correlated 2D data: variance is much larger along one direction than the other
mean = np.array([0.0, 0.0])
cov_true = np.array([[3.0, 1.5], [1.5, 1.0]])
X = rng.multivariate_normal(mean, cov_true, size=500)

X_centered = X - X.mean(axis=0)
cov_empirical = (X_centered.T @ X_centered) / (X.shape[0] - 1)

eigvals, eigvecs = np.linalg.eigh(cov_empirical)     # eigh: symmetric-matrix eigendecomposition
order = np.argsort(eigvals)[::-1]                     # sort descending: largest-variance direction first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

print("Eigenvalues (variance explained by each principal component):", np.round(eigvals, 3))
print("Fraction of variance explained by PC1:", round(eigvals[0] / eigvals.sum(), 4))

pca = PCA(n_components=2).fit(X)
print("\nsklearn PCA explained_variance_:      ", np.round(pca.explained_variance_, 3))
print("Our eigendecomposition matches sklearn:", np.allclose(sorted(eigvals), sorted(pca.explained_variance_), atol=1e-1))
print("\n(small numerical differences are expected -- sklearn PCA uses SVD internally, see 01.07)")

## 🏋️ Exercises

### Warm-up
1. By hand, find the eigenvalues of $\begin{pmatrix}2&0\\0&5\end{pmatrix}$ (hint: it's already diagonal). Then verify with `np.linalg.eig`.

### Practice
2. Implement `power_iteration` for a $4\times4$ symmetric random matrix and confirm it converges to the eigenvector associated with the LARGEST-magnitude eigenvalue reported by `np.linalg.eigh`.

### Challenge
3. Generate a positive-definite $3\times3$ matrix as $A = B^\top B$ for random $B$ (this construction guarantees positive-semi-definiteness -- explain why in a comment). Verify positive-definiteness with `is_positive_definite`, compute its condition number, then construct a second matrix with a MUCH larger condition number and show experimentally (by solving $A\mathbf{x}=\mathbf{b}$ for a slightly perturbed $\mathbf{b}$, as in Notebook 01.05) that the ill-conditioned matrix amplifies input noise far more than the well-conditioned one.

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 6 (Eigenvalues and Eigenvectors)
- [3Blue1Brown: Eigenvectors and eigenvalues](https://www.3blue1brown.com/lessons/eigenvalues)

---

*Next notebook: [SVD and PCA](07_SVD_and_PCA.ipynb)*